In [11]:
from langchain_community.utilities import SQLDatabase
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

from dotenv import load_dotenv
import os

In [12]:
load_dotenv()

api_key = os.getenv("GROQ_API_KEY")

os.environ["GROQ_API_KEY"] = api_key

In [13]:
username = "root"
password = "atharva"
host = "localhost"
database = "text_to_sql"

db = SQLDatabase.from_uri(
    f"mysql+pymysql://{username}:{password}@{host}/{database}"
)

print("Connected to DB ✅")

Connected to DB ✅


In [19]:
llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0
)

In [20]:
template = """
You are a MySQL expert.

Given the following database schema, write a SQL query that answers the question.

Rules:
- Return ONLY the SQL query
- Do NOT explain anything
- Use proper MySQL syntax
- Use backticks (`) if column names have spaces

Schema:
{schema}

Question:
{question}

SQL Query:
"""

prompt = PromptTemplate.from_template(template)

In [21]:
def get_schema(_):
    return db.get_table_info()

In [22]:
sql_chain = (
    RunnablePassthrough.assign(schema=get_schema)
    | prompt
    | llm
    | StrOutputParser()
)

In [23]:
question = "Show all tables"

response = sql_chain.invoke({"question": question})

print(response)

```sql
SHOW TABLES;
```


In [24]:
response = sql_chain.invoke({"question": "Show all tables"})
print(response)

```sql
SHOW TABLES;
```


In [25]:
def run_query(query):
    return db.run(query)

In [26]:
def ask_question(question):
    sql_query = sql_chain.invoke({"question": question})
    print("Generated SQL:", sql_query)

    result = run_query(sql_query)
    return result

In [27]:
ask_question("Show all records from Customers")

Generated SQL: SELECT * FROM customers


'[(1, \'Geiss Company\'), (2, \'Jaxbean Group\'), (3, \'Ascend Ltd\'), (4, \'Eire Corp\'), (5, \'Blogtags Ltd\'), (6, \'Family Corp\'), (7, \'Skidoo Company\'), (8, \'Amerisourc Corp\'), (9, \'Walgreen Corp\'), (10, \'Unit Ltd\'), (11, \'Voonyx Group\'), (12, \'Realcube Company\'), (13, \'Nexus Group\'), (14, \'Alembic Ltd\'), (15, \'Centizu Company\'), (16, \'Arbor Company\'), (17, \'Liberty Group\'), (18, \'Omba Group\'), (19, \'Roberts Company\'), (20, \'Tagfeed Ltd\'), (21, \'Viva Ltd\'), (22, \'SAFEWAY Ltd\'), (23, \'Devpoint Group\'), (24, \'Mylan Corp\'), (25, \'Zoonoodle Ltd\'), (26, \'Avavee Group\'), (27, \'Wise Company\'), (28, \'Army Group\'), (29, \'Voolia Ltd\'), (30, \'Shuffledri Group\'), (31, \'Golden Corp\'), (32, \'EMD Group\'), (33, \'Cadila Ltd\'), (34, \'Z.H.T. Group\'), (35, \'Aromaticos Company\'), (36, \'Trunyx Ltd\'), (37, \'Mycone Ltd\'), (38, \'Neutrogena Ltd\'), (39, \'Kazu Corp\'), (40, \'Twinte Group\'), (41, \'Pixoboo Corp\'), (42, \'Colgate-Pa Group\'),